# **Kafka configuration**

In [1]:
pip install tweepy kafka-python praw pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.8/309.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 13.7 MB/s eta 0:00:00


In [1]:
from kafka import KafkaProducer
print("KafkaProducer está disponible")

# Configuración de Kafka Producer
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

KafkaProducer está disponible


In [2]:
import praw
from kafka import KafkaProducer
import json
import time

# Configuración de Kafka Producer
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

# Reddit

In [9]:
# Configuración de Reddit API
reddit = praw.Reddit(
    client_id="L_6P81dqjOanQ6g4LtrY8A",
    client_secret="4kIkOEw-DtgPzSa6YFarZ-4UOkj6oQ",
    user_agent="reddit-sports-stream by u/SandraZhu"
)

# Subreddits deportivos
subreddits = ["soccer", "nba", "sports", "nfl", "formula1"]

print("Enviando comentarios a Kafka...")
while True:
    for sub in subreddits:
        for post in reddit.subreddit(sub).new(limit=5):
            post.comments.replace_more(limit=0)
            for comentario in post.comments[:50]:
                mensaje = {
                    "subreddit": sub,
                    "post_id": post.id,
                    "post_title": post.title,
                    "comment_id": comentario.id,
                    "comment_body": comentario.body,
                    "score": comentario.score,
                    "created_utc": comentario.created_utc
                }
                producer.send("reddit-comments", mensaje)
                print(f"Enviado comentario de r/{sub}: {comentario.body[:50]}")
    time.sleep(30)  # espera para no saturar la API



Enviando comentarios a Kafka...
Enviado comentario de r/soccer: **Mirrors / Alternative Angles**
  

*I am a bot, 
Enviado comentario de r/soccer: **Mirrors / Alternative Angles**
  

*I am a bot, 
Enviado comentario de r/soccer: Morgan Gibbs-White probably had a gun to his head 
Enviado comentario de r/soccer: Conceding a goal due to the suicidal high line who
Enviado comentario de r/soccer: when ange gets sacked christ i hope edu goes with 
Enviado comentario de r/soccer: Forest fans singing sacked in  the morning 😂😂😂Mr M
Enviado comentario de r/soccer: I’m starting to think firing Nuno might have been 
Enviado comentario de r/soccer: Great counter attack, very poor giveaway by CHO - 
Enviado comentario de r/soccer: Great counter attack, very poor giveaway by CHO - 
Enviado comentario de r/soccer: **Mirrors / Alternative Angles**
  

*I am a bot, 
Enviado comentario de r/soccer: This is such a classic Ange goal I can't say anyth
Enviado comentario de r/soccer: Nuno sends his regards


KeyboardInterrupt: 

# Youtube

In [4]:
from googleapiclient.discovery import build

# Configurar la API de YouTube
api_key = "TU_API_KEY"
youtube = build("youtube", "v3", developerKey=api_key)

video_id = "VIDEO_ID_A_MONITOREAR"

def get_comments(video_id):
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        textFormat="plainText",
        maxResults=50
    )
    response = request.execute()
    comments = []
    for item in response.get("items", []):
        snippet = item["snippet"]["topLevelComment"]["snippet"]
        comments.append({
            "source": "youtube",
            "author": snippet["authorDisplayName"],
            "text": snippet["textDisplay"],
            "published_at": snippet["publishedAt"]
        })
    return comments

while True:
    comments = get_comments(video_id)
    for c in comments:
        producer.send("youtube-comments", c)
        print("Publicado en Kafka:", c)
    time.sleep(60)  # polling cada minuto


ModuleNotFoundError: No module named 'googleapiclient'

# Twitter

In [5]:
import tweepy

bearer_token = "AAAAAAAAAAAAAAAAAAAAAJg04AEAAAAA03lI5Z2jnU2eYNSJW%2BoGmixBxa4%3D3pfVPzgSAGeXXHHIZEqXEJ3mdQ7q3YuVjulmlxyfyHSifRkPGv"
client = tweepy.Client(bearer_token=bearer_token)

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

query = "(football OR soccer OR basketball OR tennis) (#Sports OR #Game OR #Match) lang:en -is:retweet"

# Usamos paginador para obtener varios lotes
tweets = tweepy.Paginator(
    client.search_recent_tweets,
    query=query,
    max_results=100,
    limit=1  #controlas aquí el número de páginas para no gastar tu cuota
)

for page in tweets:
    for t in page.data:
        msg = {
            "source": "twitter",
            "author": None,  # gratis no da autor
            "text": t.text,
            "created_at": str(t.created_at) if hasattr(t, "created_at") else None
        }
        producer.send("twitter-comments", msg)
        print("Publicado en Kafka:", msg)


ModuleNotFoundError: No module named 'tweepy'

# PySpark

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, from_json, current_timestamp
from pyspark.sql.types import StructType, StringType, TimestampType

# Crear sesión de Spark con Delta
spark = SparkSession.builder \
    .appName("SocialMediaPipeline") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0,"
            "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Esquema común para todas las fuentes
schema = StructType() \
    .add("source", StringType()) \
    .add("author", StringType()) \
    .add("text", StringType()) \
    .add("created_at", StringType())  # vendrá como string/ISO, se puede castear

# Función genérica de lectura, limpieza y escritura
def process_stream(topic, output_path, checkpoint_path):
    # Leer de Kafka
    df = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", topic) \
        .load()

    # Parsear JSON
    parsed = df.selectExpr("CAST(value AS STRING) as json") \
        .select(from_json(col("json"), schema).alias("data")) \
        .select("data.*")

    # Limpieza básica del texto
    cleaned = parsed.withColumn(
        "clean_text",
        lower(
            regexp_replace(
                col("text"),
                r"http\S+|@\S+|#\S+|[^a-zA-ZáéíóúñÁÉÍÓÚÑ ]",
                ""
            )
        )
    ).withColumn("ingestion_time", current_timestamp())

    # Escribir en Delta Lake
    query = cleaned.writeStream \
        .format("delta") \
        .option("checkpointLocation", checkpoint_path) \
        .outputMode("append") \
        .start(output_path)

    return query


# Lanzar streams para cada fuente
reddit_query = process_stream(
    topic="reddit-comments",
    output_path="/data/delta/reddit/landing",
    checkpoint_path="/data/chk/reddit"
)

youtube_query = process_stream(
    topic="youtube-comments",
    output_path="/data/delta/youtube/landing",
    checkpoint_path="/data/chk/youtube"
)
'''
twitter_query = process_stream(
    topic="twitter-comments",
    output_path="/data/delta/twitter/landing",
    checkpoint_path="/data/chk/twitter"
)
'''
# Mantener los streams activos
spark.streams.awaitAnyTermination()


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.